# Config

In [ ]:
!pip install optuna==4.5.0

In [2]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [3]:
import pandas as pd
import os
from preprocess.preprocess import clean_text, check_deleted_expressions
from preprocess.translate import translator, gen_text_for_embedding, final_clean, detect_language
import time
import json

# 1) RoBERTa test

In [ ]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [15]:
#del gen_dataset
from utils.dataset import gen_dataset
from models.specter import embed_texts
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [ ]:
# -------------------------------
# Fine-tuning de RoBERTa en una lista de textos
# -------------------------------
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
from torch.optim import AdamW
from transformers import get_scheduler
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ===========================
# 1. Datos de ejemplo
# ===========================
"""
texts = [
    "El producto llegó en buen estado.",
    "El envío fue demasiado lento.",
    "Excelente calidad, lo recomiendo.",
    "Muy mala atención al cliente."
]
labels = [1, 0, 1, 0]   # 1 = positivo, 0 = negativo

# Dividir train/test
X_train, X_val, y_train, y_val = train_test_split(texts, labels, test_size=0.2, random_state=42)
"""
# ===========================
# 2. Tokenizador
# ===========================
model_name = "roberta-base"
tokenizer = RobertaTokenizerFast.from_pretrained(model_name)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = TextDataset(X_train, y_train, tokenizer)
test_dataset   = TextDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader   = DataLoader(test_dataset, batch_size=2)

# ===========================
# 3. Modelo
# ===========================
model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ===========================
# 4. Optimizador y scheduler
# ===========================
optimizer = AdamW(model.parameters(), lr=1e-3)
num_training_steps = len(train_loader) * 3  # 3 epochs
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

# ===========================
# 5. Entrenamiento
# ===========================
epochs = 10

model.train()
for epoch in tqdm(range(epochs)):
    model.train()
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        

    # ===========================
    # 6. Evaluación rápida
    # ===========================
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            preds = torch.argmax(outputs.logits, dim=-1)
            correct += (preds == batch["labels"]).sum().item()
            total += len(batch["labels"])



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
 10%|█         | 1/10 [00:16<02:31, 16.81s/it]

Accuracy en validación: 0.42


 20%|██        | 2/10 [00:33<02:14, 16.81s/it]

Accuracy en validación: 0.58


 30%|███       | 3/10 [00:50<01:57, 16.74s/it]

...

Accuracy en validación: 0.58


 90%|█████████ | 9/10 [02:30<00:16, 16.72s/it]

Accuracy en validación: 0.58


100%|██████████| 10/10 [02:47<00:00, 16.72s/it]

Accuracy en validación: 0.58


# 2) SPECTER test

In [4]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [5]:
#del gen_dataset
from utils.dataset import gen_dataset
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

## Sin criterion

In [ ]:
# Fine-tuning de SPECTER (allenai/specter) para clasificación
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from tqdm import tqdm

"""
# ---------- Datos de ejemplo ----------
texts = [
    "Deep learning improves medical image classification.",
    "We propose a new finite element method for biomechanics.",
    "A survey on NLP for legal documents.",
    "Spectral methods for time series forecasting."
]

labels = [1, 1, 0, 0]  # ej. 2 clases
X_train, X_val, y_train, y_val = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)
"""
# Si tienes título y abstract, arma algo así:
# texts = [f"{title} [SEP] {abstract}" for title, abstract in data]

# ---------- Tokenizador y Dataset ----------
model_name = "allenai/specter"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.enc = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = TextDataset(X_train, y_train, tokenizer)
val_ds   = TextDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False)

# ---------- Modelo (capa de clasificación encima de SPECTER) ----------
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ---------- Optimizador y scheduler ----------
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
epochs = 10
num_training_steps = len(train_loader) * epochs
scheduler = get_scheduler(
    "linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

# ---------- Loop de entrenamiento ----------
for epoch in range(epochs):
    model.train()
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for batch in loop:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(**batch)
        loss = out.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        loop.set_postfix(loss=loss.item())

    # ---------- Evaluación rápida ----------
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            preds = out.logits.argmax(dim=-1)
            correct += (preds == batch["labels"]).sum().item()
            total += batch["labels"].size(0)
    print(f"Val acc: {correct/total:.3f}")

# (Opcional) Guardar modelo
# model.save_pretrained("./specter_cls")
# tokenizer.save_pretrained("./specter_cls")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/specter and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1/10: 100%|██████████| 97/97 [00:12<00:00,  7.66it/s, loss=1.07] 


Val acc: 0.648


Epoch 2/10: 100%|██████████| 97/97 [00:12<00:00,  7.66it/s, loss=0.469]


Val acc: 0.653


Epoch 3/10: 100%|██████████| 97/97 [00:12<00:00,  7.60it/s, loss=0.0763]


...

Val acc: 0.710


Epoch 9/10: 100%|██████████| 97/97 [00:12<00:00,  7.51it/s, loss=0.00525]


Val acc: 0.710


Epoch 10/10: 100%|██████████| 97/97 [00:12<00:00,  7.54it/s, loss=0.00167]


Val acc: 0.710


## Con criterion

In [6]:
# Fine-tuning de SPECTER (allenai/specter) para clasificación
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from tqdm import tqdm

def get_sample_weights_loss(y):
  y = np.asarray(y, dtype=np.int64)
  class_counts = np.bincount(y)
  class_weights = 1.0 / class_counts
  class_weights = class_weights / class_weights.sum()

  return class_weights

"""
# ---------- Datos de ejemplo ----------
texts = [
    "Deep learning improves medical image classification.",
    "We propose a new finite element method for biomechanics.",
    "A survey on NLP for legal documents.",
    "Spectral methods for time series forecasting."
]

labels = [1, 1, 0, 0]  # ej. 2 clases
X_train, X_val, y_train, y_val = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)
"""
# Si tienes título y abstract, arma algo así:
# texts = [f"{title} [SEP] {abstract}" for title, abstract in data]

# ---------- Tokenizador y Dataset ----------
model_name = "allenai/specter"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.enc = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = TextDataset(X_train, y_train, tokenizer)
val_ds   = TextDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False)

# ---------- Modelo (capa de clasificación encima de SPECTER) ----------
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ---------- Optimizador y scheduler ----------
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
epochs = 10
num_training_steps = len(train_loader) * epochs
scheduler = get_scheduler(
    "linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)
class_weights = get_sample_weights_loss(y_train)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

# ---------- Loop de entrenamiento ----------
for epoch in range(epochs):
    model.train()
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for batch in loop:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(**{k: v for k, v in batch.items() if k != "labels"})
        logits = out.logits
        loss = criterion(logits, batch["labels"].to(device))
        loss.backward()
        optimizer.step()
        scheduler.step()
        loop.set_postfix(loss=loss.item())

    # ---------- Evaluación rápida ----------
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            preds = out.logits.argmax(dim=-1)
            correct += (preds == batch["labels"]).sum().item()
            total += batch["labels"].size(0)
    print(f"Val acc: {correct/total:.3f}")


/usr/local/lib/python3.10/dist-packages/torch/_utils.py:830: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/specter and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1/10: 100%|██████████| 193/193 [00:14<00:00, 12.92it/s, loss=0.651]


Val acc: 0.663


Epoch 2/10: 100%|██████████| 193/193 [00:14<00:00, 13.12it/s, loss=0.312]


Val acc: 0.663


Epoch 3/10: 100%|██████████| 193/193 [00:14<00:00, 13.15it/s, loss=0.144] 


...

Val acc: 0.731


Epoch 9/10: 100%|██████████| 193/193 [00:14<00:00, 13.01it/s, loss=0.00102] 


Val acc: 0.715


Epoch 10/10: 100%|██████████| 193/193 [00:14<00:00, 12.89it/s, loss=0.00154] 


Val acc: 0.715


# 3) SPECTER 

## Functions

In [116]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import os
import numpy as np
from collections import Counter
import copy
import inspect
from torch.utils.data import Dataset
from torch.optim import AdamW

def get_sample_weights_loss(y):
  y = np.asarray(y, dtype=np.int64)
  class_counts = np.bincount(y)
  class_weights = 1.0 / class_counts
  class_weights = class_weights / class_weights.sum()

  return class_weights

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.enc = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class Pytorch_Pipeline():
    def __init__(self, model_class, sample_weights_loss=None, max_epochs = 200, use_scheduler=None):
        #Set device
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        #Modelo
        self.model_class = model_class
        self.model = None
        #Elementos del entrenamiento
        self.params = None
        self.sample_weights_loss = sample_weights_loss
        self.criterion = None
        self.optimizer = None
        self.batch_size = None
        self.scheduler=None
        self.max_epochs = max_epochs
        #scheduler
        self.use_scheduler=use_scheduler
        #Best model
        self.best_model_state=None

    def partial_fit(self, loader):
        self.model.to(self.device)
        self.model.train()
        
        for batch in loader:
            batch = {k: v.to(self.device) for k, v in batch.items()}
            self.optimizer.zero_grad()
            out = model(**{k: v for k, v in batch.items() if k != "labels"})
            logits = out.logits
            loss = self.criterion(logits, batch["labels"].to(self.device))
            loss.backward()
            self.optimizer.step()
            if self.use_scheduler is not None:
                self.scheduler.step()

        return self
        
    def predict_and_evaluate(self, loader):
            self.model.eval()
            total_loss = 0.0
            total_samples = 0
            all_preds, all_targets = [], []

            with torch.no_grad():
                for batch in loader:
                    # mover batch al device
                    batch = {k: v.to(self.device) for k, v in batch.items()}
                    xb = {k: v for k, v in batch.items() if k != "labels"}
                    yb = batch["labels"]

                    outputs = self.model(**xb)
                    logits = outputs.logits

                    # calcular pérdida (soporta reduction='mean' o 'none')
                    loss_val = self.criterion(logits, yb)
                    if loss_val.dim() > 0:              # p.ej., reduction='none' -> [B]
                        batch_loss = loss_val.mean()
                    else:
                        batch_loss = loss_val

                    bs = yb.size(0)
                    total_loss += batch_loss.item() * bs  # acumular ponderado por tamaño de batch
                    total_samples += bs

                    # predicciones
                    preds = logits.argmax(dim=1)

                    all_preds.append(preds.cpu())
                    all_targets.append(yb.cpu())

            avg_val_loss = total_loss / max(total_samples, 1)
            y_true = torch.cat(all_targets).numpy()
            y_pred = torch.cat(all_preds).numpy()
            f1 = f1_score(y_true, y_pred, average='weighted')

            return avg_val_loss, f1, y_true, y_pred

    def set_params(self, **params):
          self.params = params

          # Obtener los parámetros esperados por el constructor de model_class
          signature = inspect.signature(self.model_class.__init__)
          valid_keys = set(signature.parameters.keys()) - {'self'}

          # Filtrar los params para incluir solo los esperados
          filtered_params = {k: v for k, v in params.items() if k in valid_keys}

          #self.model = self.model_class(**filtered_params)
          self.model = self.model_class
          self.optimizer = AdamW(self.model.parameters(), lr=self.params['lr']) 
          self.batch_size = self.params['batch_size']

    def set_criterion(self, y):
          # ----------- Criterion -----------
          if self.sample_weights_loss is not None:
              class_weights = get_sample_weights_loss(y)
              class_weights = torch.tensor(class_weights, dtype=torch.float32).to(self.device)
              self.criterion = nn.CrossEntropyLoss(weight=class_weights)
          else:
              self.criterion = nn.CrossEntropyLoss()

          return self

    def fit_early_stopping(self, train_loader, val_loader, labels):
        #Establecer criterion con sample weights si se especifica
        self.set_criterion(labels)
        self.best_model_state = None
        # ---------- Early stopping (por pérdida) ----------
        patience = 10
        min_delta = 1e-4
        best_val_loss = float('inf')
        epochs_no_improve = 0
        #scheduler
        num_training_steps = len(train_loader) * self.max_epochs
        if self.use_scheduler is not None:
            self.scheduler = get_scheduler(
                "linear", optimizer=self.optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
            )
        #Entrenamiento
        for epoch in range(self.max_epochs):
            self.partial_fit(train_loader)
            avg_val_loss, f1, _, _ = self.predict_and_evaluate(val_loader)
            
            if avg_val_loss + min_delta < best_val_loss:
                best_val_loss = avg_val_loss
                self.best_model_state = self.model.state_dict()
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    break
            print("f1:", f1)
        return f1
    
    def eval_test(self, model_dict, loader):
        all_preds, all_targets = [], []
        model = self.model_class
        model.load_state_dict(model_dict)
        with torch.no_grad():
            for batch in loader:
                # mover batch al device
                batch = {k: v.to(self.device) for k, v in batch.items()}
                xb = {k: v for k, v in batch.items() if k != "labels"}
                yb = batch["labels"]

                outputs = model(**xb)
                logits = outputs.logits

                # predicciones
                preds = logits.argmax(dim=1)

                all_preds.append(preds.cpu())
                all_targets.append(yb.cpu())

        y_true = torch.cat(all_targets).numpy()
        y_pred = torch.cat(all_preds).numpy()
        metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
            'recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
            'f1_score': f1_score(y_true, y_pred, average='macro'),
            'cm': confusion_matrix(y_true, y_pred)
        }
        return metrics

In [117]:
#del gen_dataset
from utils.dataset import gen_dataset
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from torch.optim import AdamW

model_name = "allenai/specter"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_ds = TextDataset(X_train, y_train, tokenizer)
val_ds   = TextDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False)

# ---------- Modelo (capa de clasificación encima de SPECTER) ----------
model = AutoModelForSequenceClassification.from_pretrained(model_name)
pipeline = Pytorch_Pipeline(model_class=model, use_scheduler=None)

params={
    "lr": 5e-5,
    "batch_size":4
}
pipeline.set_params(**params)
pipeline.fit_early_stopping(train_loader, val_loader, y_train)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/specter and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Code

In [ ]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [ ]:
#del gen_dataset
from utils.dataset import gen_dataset
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)



In [ ]:
from utils.dataset import CvCustom

#Definir variables
model_name = "allenai/specter"
cv_function=CvCustom(df_decode)
X_train=np.array(X_train)

params={
    "lr": 5e-5,
    "batch_size":4
    }

#Train
results = []
for train_idx, test_idx in cv_function.split(X_train):
    #Split data
    xt, yt = X_train[train_idx], y_train[train_idx]
    xv, yv = X_train[test_idx], y_train[test_idx]
    #define tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    #Datasets
    train_ds = TextDataset(list(xt), yt, tokenizer)
    val_ds   = TextDataset(list(xv), yv, tokenizer)
    test_ds = TextDataset(list(X_test), y_test, tokenizer)
    #Loaders
    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=8, shuffle=False)
    # ---------- Modelo (capa de clasificación encima de SPECTER) ----------
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    pipeline = Pytorch_Pipeline(model_class=model, use_scheduler=True, max_epochs=200)
    #Train
    pipeline.set_params(**params)
    pipeline.fit_early_stopping(train_loader, val_loader, yt)
    #Get test results
    metrics = pipeline.eval_test(pipeline.best_model_state, test_loader)
    #Save results
    print(metrics)
    results.append(metrics["f1_score"])

mean=np.mean(results)
std=np.std(results)
print("mean:", mean)
print("std:", std)


# 3) SPECTER OPTUNA

## Functions

### Pytorch pipeline

In [76]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import os
import numpy as np
from collections import Counter
import copy
import inspect
from torch.utils.data import Dataset
from torch.optim import AdamW

def get_sample_weights_loss(y):
  y = np.asarray(y, dtype=np.int64)
  class_counts = np.bincount(y)
  class_weights = 1.0 / class_counts
  class_weights = class_weights / class_weights.sum()

  return class_weights

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.enc = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class Pytorch_Pipeline():
    def __init__(self, model_class, sample_weights_loss=None, max_epochs = 200, use_scheduler=None):
        #Set device
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        #Modelo
        self.model_class = model_class
        self.model = None
        #Elementos del entrenamiento
        self.params = None
        self.sample_weights_loss = sample_weights_loss
        self.criterion = None
        self.optimizer = None
        self.batch_size = None
        self.scheduler=None
        self.max_epochs = max_epochs
        #scheduler
        self.use_scheduler=use_scheduler
        #Best model
        self.best_model_state=None

    def partial_fit(self, loader):
        self.model.to(self.device)
        self.model.train()
        
        for batch in loader:
            batch = {k: v.to(self.device) for k, v in batch.items()}
            self.optimizer.zero_grad()
            out = self.model(**{k: v for k, v in batch.items() if k != "labels"})
            logits = out.logits
            loss = self.criterion(logits, batch["labels"].to(self.device))
            loss.backward()
            self.optimizer.step()
            if self.use_scheduler is not None:
                self.scheduler.step()

        return self
            
    def predict_and_evaluate(self, loader):
            self.model.eval()
            total_loss = 0.0
            total_samples = 0
            all_preds, all_targets = [], []

            with torch.no_grad():
                for batch in loader:
                    # mover batch al device
                    batch = {k: v.to(self.device) for k, v in batch.items()}
                    xb = {k: v for k, v in batch.items() if k != "labels"}
                    yb = batch["labels"]

                    outputs = self.model(**xb)
                    logits = outputs.logits

                    # calcular pérdida (soporta reduction='mean' o 'none')
                    loss_val = self.criterion(logits, yb)
                    if loss_val.dim() > 0:              # p.ej., reduction='none' -> [B]
                        batch_loss = loss_val.mean()
                    else:
                        batch_loss = loss_val

                    bs = yb.size(0)
                    total_loss += batch_loss.item() * bs  # acumular ponderado por tamaño de batch
                    total_samples += bs

                    # predicciones
                    preds = logits.argmax(dim=1)

                    all_preds.append(preds.cpu())
                    all_targets.append(yb.cpu())

            avg_val_loss = total_loss / max(total_samples, 1)
            y_true = torch.cat(all_targets).numpy()
            y_pred = torch.cat(all_preds).numpy()
            f1 = f1_score(y_true, y_pred, average='weighted')

            return avg_val_loss, f1, y_true, y_pred

    def set_params(self, **params):
        self.params = params

        # Obtener los parámetros esperados por el constructor de model_class
        signature = inspect.signature(self.model_class.__init__)
        valid_keys = set(signature.parameters.keys()) - {'self'}

        # Filtrar los params para incluir solo los esperados
        filtered_params = {k: v for k, v in params.items() if k in valid_keys}

        #self.model = self.model_class(**filtered_params)
        
        self.model = self.model_class
        if torch.cuda.device_count() > 1:
            print("Usando", torch.cuda.device_count(), "GPUs")
            self.model = torch.nn.DataParallel(self.model)
        self.optimizer = AdamW(self.model.parameters(), lr=self.params['lr']) 
        self.batch_size = self.params['batch_size']

        # Si el modelo está envuelto en DataParallel, accedemos al .module
        model_to_unfreeze = self.model.module if isinstance(self.model, torch.nn.DataParallel) else self.model
        if self.params.get("n_unfreeze") is not None:
            for layer in model_to_unfreeze.bert.encoder.layer[-self.params["n_unfreeze"]:]:
                for p in layer.parameters():
                    p.requires_grad = True
              
    def set_criterion(self, y):
          # ----------- Criterion -----------
          if self.sample_weights_loss is not None:
              class_weights = get_sample_weights_loss(y)
              class_weights = torch.tensor(class_weights, dtype=torch.float32).to(self.device)
              self.criterion = nn.CrossEntropyLoss(weight=class_weights)
          else:
              self.criterion = nn.CrossEntropyLoss()

          return self

    def fit_early_stopping(self, train_loader, val_loader, labels):
        #Establecer criterion con sample weights si se especifica
        self.set_criterion(labels)
        self.best_model_state = None
        # ---------- Early stopping (por pérdida) ----------
        patience = 10
        min_delta = 1e-4
        best_val_loss = float('inf')
        epochs_no_improve = 0
        #scheduler
        num_training_steps = len(train_loader) * self.max_epochs
        if self.use_scheduler is not None:
            self.scheduler = get_scheduler(
                "linear", optimizer=self.optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
            )
        #Entrenamiento
        for epoch in range(self.max_epochs):
            self.partial_fit(train_loader)
            avg_val_loss, f1, _, _ = self.predict_and_evaluate(val_loader)
            
            if avg_val_loss + min_delta < best_val_loss:
                best_val_loss = avg_val_loss
                self.best_model_state = self.model.state_dict()
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    break
            print("f1:", f1)
        return f1
    
    def eval_test(self, model_dict, loader):
        all_preds, all_targets = [], []
        model = self.model_class
        model.load_state_dict(model_dict)
        with torch.no_grad():
            for batch in loader:
                # mover batch al device
                batch = {k: v.to(self.device) for k, v in batch.items()}
                xb = {k: v for k, v in batch.items() if k != "labels"}
                yb = batch["labels"]

                outputs = model(**xb)
                logits = outputs.logits

                # predicciones
                preds = logits.argmax(dim=1)

                all_preds.append(preds.cpu())
                all_targets.append(yb.cpu())

        y_true = torch.cat(all_targets).numpy()
        y_pred = torch.cat(all_preds).numpy()
        metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
            'recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
            'f1_score': f1_score(y_true, y_pred, average='macro'),
            'cm': confusion_matrix(y_true, y_pred)
        }
        return metrics

### Optuna

In [79]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import optuna
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np


def convert_numpy_to_native(obj):
    if isinstance(obj, dict):
        return {k: convert_numpy_to_native(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_to_native(x) for x in obj]
    elif isinstance(obj, np.generic):  # np.float64, np.int64, etc.
        return obj.item()
    else:
        return obj

def get_metrics(y_true, y_pred, verbose = True):
  metrics = {
      'accuracy': accuracy_score(y_true, y_pred),
      'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
      'recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
      'f1_score': f1_score(y_true, y_pred, average='weighted'),
      'cm': confusion_matrix(y_true, y_pred)
  }
  if verbose:
    print(metrics)
  return metrics

class optuna_objective_cv:
    def __init__(self, X, y, n_classes, model_name, SMOTE_on=None, sample_weights_loss=None, Test_mode = None):
        self.results = {}
        self.X = X
        self.y = y
        self.n_classes = n_classes
        self.sample_weights_loss = sample_weights_loss
        self.max_epochs = 200
        self.best_model_trial = None
        self.Test_mode = Test_mode
        #BERT models
        self.model_name = model_name
    
    def get_loaders(self, X_train, X_test, y_train, y_test, batch_size):
        train_dataset = TextDataset(list(X_train), y_train, self.tokenizer)
        train_loader = DataLoader(train_dataset, batch_size, shuffle=True)

        test_dataset = TextDataset(list(X_test), y_test, self.tokenizer)
        test_loader = DataLoader(test_dataset, batch_size, shuffle=False)

        return train_loader, test_loader

    def objective(self, trial):
        # ----------- Hiperparámetros a optimizar -----------
        params={
        "lr": trial.suggest_float("lr", 1e-5, 5e-5, log=True),
        "batch_size":12,
        "n_unfreeze":trial.suggest_int("n_unfreeze", 6, 12)
        }
    
        #------------- StratifiedKFold -------------------------------
        F1 = []
        all_metrics = []
        cv_function=CvCustom(df_decode)
        for fold, (train_index, test_index) in enumerate(cv_function.split(self.X)):
            #---------------Split data-------------------------------
            X_train, X_test = self.X[train_index], self.X[test_index]
            y_train, y_test = self.y[train_index], self.y[test_index]
            #--------------def model----------------------------------
            model = AutoModelForSequenceClassification.from_pretrained(self.model_name)
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            pipeline_mlp =  Pytorch_Pipeline(model_class=model, sample_weights_loss = self.sample_weights_loss)
            #Set params
            pipeline_mlp.set_params(**params)
            #Set criterion
            pipeline_mlp.set_criterion(y_train)

            # ------------- Loaders --------------------
            train_loader, test_loader = self.get_loaders(X_train, X_test, y_train, y_test, pipeline_mlp.batch_size)
            # ---------- Early stopping (por loss) ----------
            patience = 10
            min_delta = 1e-4
            best_val_loss = float('inf')
            epochs_no_improve = 0
            best_model_state = None

            for epoch in range(pipeline_mlp.max_epochs):
                pipeline_mlp.partial_fit(train_loader)
                avg_val_loss, f1, y_test, y_pred = pipeline_mlp.predict_and_evaluate(test_loader)
                # ---------- Optuna pruning con F1 ----------
                #Prune only on the first fold
                if fold == 0:
                    trial.report(f1, epoch)
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()
                # ---------- Early stopping (por loss) ----------
                if avg_val_loss + min_delta < best_val_loss:
                    best_val_loss = avg_val_loss
                    best_model_state = pipeline_mlp.model.state_dict()
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1
                    if epochs_no_improve >= patience:
                        break

            #---------------- Save final result ----------------
            F1.append(f1)
            #-------------Visualization metrics-----------------
            metrics = get_metrics(y_test, y_pred)
            all_metrics.append(metrics)

        #------------ Compute avg among 5 folds ----------------
        mean_F1 = np.mean(F1)

        # ---------- Guarda el modelo del mejor trial según F1 ----------
        try:
            if trial.number == 0 or mean_F1 > trial.study.best_value:
                self.results = {
                    'metrics': self.avg_metrics(all_metrics),
                    'best_params': params,
                    'model_state_dict': best_model_state,
                    'epoch_number': epoch
                }

        except ValueError:
          pass

        return mean_F1
    
    def avg_metrics(self, all_metrics):
        avg_metrics = {}
        for metric in all_metrics[0].keys():
            values = [metrics[metric] for metrics in all_metrics]

            if metric == 'cm':
                avg_metrics[metric] = np.mean(values, axis=0).astype(int)  # o float si prefieres
            else:
                avg_metrics[metric] = np.mean(values)
        return avg_metrics

    #----------- Método para obtener los resultados -----------
    def get_results(self):
      return self.results

## Code

In [80]:
#del gen_dataset
from utils.dataset import gen_dataset
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from utils.dataset import CvCustom

model_name = "allenai/specter"
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# ----------- Lanzar la optimización -----------
study = optuna.create_study(direction="maximize")
X_train=np.array(X_train)
tokenizer = AutoTokenizer.from_pretrained(model_name)
opt_model = optuna_objective_cv(X_train, y_train, n_classes=2, model_name = model_name,
                                sample_weights_loss=True, Test_mode=True)
study.optimize(opt_model.objective, n_trials=20)

# ----------- Mostrar mejores resultados -----------
print("Mejor f1-score:", study.best_value)
print("Mejores hiperparámetros:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

#Read results from the best model
results_best_model = opt_model.get_results()

# Aplica la conversión
metrics_native = convert_numpy_to_native(results_best_model['metrics'])

# Imprime con formato limpio
print({'metrics': metrics_native})

In [82]:
from utils.dataset import CvCustom

#Definir variables
model_name = "allenai/specter"
cv_function=CvCustom(df_decode)
X_train=np.array(X_train)

params={
    "lr": 5e-5,
    "batch_size":4,
    "n_unfreeze":12 #12 max
    }

#Train
results = []
for train_idx, test_idx in cv_function.split(X_train):
    #Split data
    xt, yt = X_train[train_idx], y_train[train_idx]
    xv, yv = X_train[test_idx], y_train[test_idx]
    #define tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    #Datasets
    train_ds = TextDataset(list(xt), yt, tokenizer)
    val_ds   = TextDataset(list(xv), yv, tokenizer)
    test_ds = TextDataset(list(X_test), y_test, tokenizer)
    #Loaders
    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=8, shuffle=False)
    # ---------- Modelo (capa de clasificación encima de SPECTER) ----------
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    pipeline = Pytorch_Pipeline(model_class=model, use_scheduler=False, max_epochs=200)
    #Train
    pipeline.set_params(**params)
    pipeline.fit_early_stopping(train_loader, val_loader, yt)
    #Get test results
    metrics = pipeline.eval_test(pipeline.best_model_state, test_loader)
    #Save results
    print(metrics)
    results.append(metrics["f1_score"])

mean=np.mean(results)
std=np.std(results)
print("mean:", mean)
print("std:", std)

(771,)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/specter and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Usando 2 GPUs
f1: 0.5623645024166521
f1: 0.6156027044843336
f1: 0.5472142211441824
f1: 0.5450798131617541
f1: 0.6062336101183659
f1: 0.5383290673280517
f1: 0.6000950338748279
f1: 0.6045379067634886
f1: 0.5820851850559966
f1: 0.5797700195572432


RuntimeError: Error(s) in loading state_dict for BertForSequenceClassification:
	Missing key(s) in state_dict: "bert.embeddings.word_embeddings.weight", "bert.embeddings.position_embeddings.weight", "bert.embeddings.token_type_embeddings.weight", "bert.embeddings.LayerNorm.weight", "bert.embeddings.LayerNorm.bias", "bert.encoder.layer.0.attention.self.query.weight", "bert.encoder.layer.0.attention.self.query.bias", "bert.encoder.layer.0.attention.self.key.weight", "bert.encoder.layer.0.attention.self.key.bias", "bert.encoder.layer.0.attention.self.value.weight", "bert.encoder.layer.0.attention.self.value.bias", "bert.encoder.layer.0.attention.output.dense.weight", "bert.encoder.layer.0.attention.output.dense.bias", "bert.encoder.layer.0.attention.output.LayerNorm.weight", "bert.encoder.layer.0.attention.output.LayerNorm.bias", "bert.encoder.layer.0.intermediate.dense.weight", "bert.encoder.layer.0.intermediate.dense.bias", "bert.encoder.layer.0.output.dense.weight", "bert.encoder.layer